# Timestamped Transcription of Bengali YouTube Videos

### Model Setup

In [ ]:
import ctypes

for dll in [
    r"venv\Lib\site-packages\nvidia\cublas\bin\cublas64_12.dll",
    r"venv\Lib\site-packages\nvidia\cudnn\bin\cudnn64_9.dll",
]:
    ctypes.CDLL(dll)

print("CUDA DLLs loaded")


CUDA DLLs loaded


### Transcription Helper Function

In [3]:
import jiwer
import os
import csv
import string
import re
import time
import pandas as pd
from IPython.display import display

def stream_chunks(segments, max_gap=1.0, max_words=5):
    words, start, end, previous_end = [], None, None, None
    for segment in segments:
        for word in segment.words:
            text = word.word.strip()
            should_split = words and (
                word.start - previous_end > max_gap
                or words[-1].endswith((".", "?", "!"))
                or len(words) >= max_words
            )
            if should_split:
                yield {"start": start, "end": end, "text": " ".join(words)}
                words, start = [], None
            if not words:
                start = word.start
            words.append(text)
            end = previous_end = word.end
    if words:
        yield {"start": start, "end": end, "text": " ".join(words)}

def calculate_metrics(gt_text, pred_text, title):
    gt_clean = re.sub(r'\s+', ' ', gt_text.replace('\n', ' ')).translate(str.maketrans('', '', string.punctuation + '।')).strip()
    pred_clean = re.sub(r'\s+', ' ', pred_text.replace('\n', ' ')).translate(str.maketrans('', '', string.punctuation + '।')).strip()
    
    if len(gt_clean) == 0:
        return None
        
    out = jiwer.process_words(gt_clean, pred_clean)
    cer = jiwer.cer(gt_clean, pred_clean)
    gt_words = len(gt_clean.split())
    
    return {
        "Video": title,
        "WER %": round(out.wer * 100, 2),
        "CER %": round(cer * 100, 2),
        "Word Accuracy %": round((out.hits / gt_words) * 100, 2) if gt_words > 0 else 0,
        "GT Words": gt_words,
        "Pred Words": len(pred_clean.split()),
        "Correct Words": out.hits,
        "Substitutions": out.substitutions,
        "Deletions (Missed)": out.deletions,
        "Insertions": out.insertions
    }


def run_transcription(batched_model, model_name, dataset_dir=r"dataset", output_dir=r"Outputs", **kwargs):
    os.makedirs(output_dir, exist_ok=True)
    metadata = []
    if os.path.exists(os.path.join(dataset_dir, "metadata.csv")):
        with open(os.path.join(dataset_dir, "metadata.csv"), 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                metadata.append(row)

    results = []
    for row in metadata:
        title = row['Title']
        safe_title = "".join([c for c in title if c.isalnum() or c in (' ', '-', '_')]).strip().replace(' ', '_')
        audio_path = os.path.join(dataset_dir, row['Audio_File'])
        subtitle_path = os.path.join(dataset_dir, row['Subtitle_File'])
        
        if not os.path.exists(audio_path) or not os.path.exists(subtitle_path):
            continue
            
        print(f"\nTranscribing {title} with {model_name}...")
        
        start_time = time.time()
        
        transcribe_args = {
            "batch_size": 16,
            "vad_filter": True,
            "word_timestamps": True
        }
        transcribe_args.update(kwargs)
        
        segments, info = batched_model.transcribe(audio_path, **transcribe_args)

        chunks = list(stream_chunks(segments))

        print("Streaming Chunked Output:")
        output_file_content = "Streaming Chunked Output:\n"
        pred_text_raw = ""
        
        for chunk in chunks:
            line = f"{chunk['start']:.2f} - {chunk['end']:.2f}: {chunk['text']}"
            print(line)
            output_file_content += line + "\n"
            pred_text_raw += chunk['text'] + " "

        time_taken = time.time() - start_time
        footer = f"\nTranscription & Chunking took {time_taken:.2f} seconds\nTotal chunks generated: {len(chunks)}\n"
        print(footer)
        output_file_content += footer
        
        out_file = os.path.join(output_dir, f"{model_name}_{safe_title}.txt")
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(output_file_content)
            
        with open(subtitle_path, 'r', encoding='utf-8') as f:
            gt_text = f.read()
            
        metrics = calculate_metrics(gt_text, pred_text_raw, title)
        if metrics:
            results.append(metrics)

    if results:
        df = pd.DataFrame(results)
        display(df)



## Loading Model & Transcribing

### Faster-whisper-large-v3

In [2]:
from faster_whisper import WhisperModel, BatchedInferencePipeline

model = WhisperModel("large-v3", device="cuda", compute_type="float16")
batched_model = BatchedInferencePipeline(model=model)

print("Loaded batched model successfully")


Loaded batched model successfully


In [5]:
run_transcription(
    batched_model=batched_model,
    model_name="Faster-whisper-large-v3",
    language='bn',
    vad_parameters={"min_silence_duration_ms": 4000, "speech_pad_ms": 6000}
)



Transcribing Rain with Faster-whisper-large-v3...
Streaming Chunked Output:
0.00 - 2.08: দোখিন্বঙ্গে আজ ও কাল ভারি
2.08 - 4.52: বৃষ্টির সম্ভাবনা কলকাতায় দফায় দফায়
4.52 - 6.78: বৃষ্টির সম্ভাবনা থাকছে শুক্রুবার থেকে
6.78 - 7.38: এই বৃষ্টি �
16.65 - 19.57: অন্ন জেলাতেও বিখিপ্ত দোযাক পাসলা
19.57 - 22.39: বৃষ্টির সম্বাবনা থাকছে শোনি এবং
22.39 - 24.91: রবিবার দোখিন বঙ্গে বৃষ্টির পরিমান
24.91 - 41.93: কোম্বে আর অন�
44.01 - 47.05: এমোটে মরসুমে অখ্যরেখা মরশিদাবাদ থেকে
47.05 - 49.33: গাঙ্গিয় দক্ষিন মঙ্গের প্রায় অপর
49.33 - 51.37: দিয়ে বাংলাদেশ পরজন্ত বিস্তিতো সেই
51.37 - 63.85: মরসু� 30 থেকে 40 কিলমিটের
63.85 - 65.39: বেগে দমকা জহরা হাওয়ার পুর্বা
65.39 - 67.21: ভা শুলো এছে উপকুলের জিলার
67.21 - 68.97: খেত্রে এই দমকা জহরা হাওয়ার
68.97 - 70.27: গতিবেক কোনকনো �
77.40 - 79.66: এই অবস্থা বৃহস্পতিবার পরজন্ত চলার
79.66 - 81.60: পর শুক্র, সনি এবং রবি,
81.68 - 83.92: উত্তর এবং দখিন বঙ্গে লক্খনিয়
83.92 - 107.38: ভাবে বৃষ্টি পাতের পরিমান কোম�
108.51 - 112.11: BANGALORE AWACH, BANGALI RABIC, ZEE
1

,Video,WER %,CER %,Word Accuracy %,GT Words,Pred Words,Correct Words,Substitutions,Deletions (Missed),Insertions
0,Rain,81.40,65.57,20.47,215,101,44,53,118,4
1,Delhi Protest,83.87,62.55,17.74,186,98,33,62,91,3
2,Subhendu CM,87.22,71.46,13.10,626,227,82,143,401,2
3,Dengue,86.54,65.26,13.74,364,191,50,140,174,1
4,Terrorist(Long),83.46,64.16,17.49,2527,1153,442,687,1398,24
5,Taslima Nasrin(Long),78.28,56.14,22.43,2670,1472,599,854,1217,19
6,Shamik(Long),83.31,63.13,17.56,2625,1200,461,716,1448,23
7,10am News(Long),86.77,66.44,14.21,2456,1025,349,652,1455,24


### Faster-whisper-large-v3-bn-ct2

In [2]:
from faster_whisper import WhisperModel, BatchedInferencePipeline

model = WhisperModel("models\\whisper-large-v3-bn-ct2", device="cuda", compute_type="float16")

batched_model = BatchedInferencePipeline(model=model)

print("Loaded batched model successfully")

Loaded batched model successfully


In [4]:
run_transcription(
    batched_model=batched_model,
    model_name="Faster-whisper-large-v3-bn",
    language='bn',
    vad_parameters={"min_silence_duration_ms": 4000, "speech_pad_ms": 6000}
)



Transcribing Rain with Faster-whisper-large-v3-bn...
Streaming Chunked Output:
0.00 - 2.14: দক্ষিণবঙ্গে, আজ ও কাল ভারী
2.14 - 4.52: বৃষ্টির সম্ভাবনা কলকাতায় দফায় দফায়
4.52 - 6.96: বৃষ্টির সম্ভাবনা থাকছে।শুক্রবার থেকে এই
6.96 - 18.83: বৃষ্টি ক অন্য জেলাতেও বিক্ষিপ্ত
18.83 - 22.01: দুয়াগপশলা বৃষ্টির সম্ভাবনা থাকছে, সোনি
22.01 - 24.91: এবং রবিবার দক্ষিণবঙ্গে বৃষ্টির পরিমান
24.91 - 27.99: কমবে, আর অন�
43.71 - 46.71: এই মতে মৌসুমী অক্ষরেখা মরশিদাবাদ
46.71 - 49.39: থেকে গাঙ্গীয় দক্ষিণবঙ্গের প্রায় উপর
49.39 - 53.39: দিয়ে বাংলাদেশ পর্যন্ত বিস্তৃত সেই
53.39 - 63.85: মৌসুম তিরিস থেকে চল্লিশ কিলোমিটার
63.85 - 65.53: বেগে দমকা ঝোড়া হওয়ার পূর্বাভাস
65.53 - 68.39: রয়েছে।উপকূলের জেলার ক্ষেত্রে এই দমকা
68.39 - 78.76: ঝোড়া হওয়া এই অবস্থা বৃহস্পতিবার
78.76 - 81.04: পর্যন্ত চলার পর শুক্র, সনি
81.04 - 83.30: এবং রবি উত্তর এবং দক্ষিণবঙ্গে
83.30 - 107.38: লক্ষণীয়ভাবে বৃষ্টিপাতের পরিমান কম
108.43 - 112.09: বাংলার আওয়াজ বাঙালির আবেগ সি
112.09 - 112.89: চব্বিশ ঘটা

Transcription & Chunking took 

,Video,WER %,CER %,Word Accuracy %,GT Words,Pred Words,Correct Words,Substitutions,Deletions (Missed),Insertions
0,Rain,70.70,60.75,29.77,215,89,64,24,127,1
1,Delhi Protest,63.98,52.34,37.10,186,94,69,23,94,2
2,Subhendu CM,74.76,66.82,25.56,626,233,160,71,395,2
3,Dengue,70.05,57.35,31.04,364,178,113,61,190,4
4,Terrorist(Long),69.57,58.17,31.14,2527,1164,787,359,1381,18
5,Taslima Nasrin(Long),61.91,49.23,39.29,2670,1491,1049,410,1211,32
6,Shamik(Long),72.19,59.64,28.46,2625,1161,747,397,1481,17
7,10am News(Long),70.56,60.12,30.01,2456,1070,737,319,1400,14


### Faster-whisper-tugstugi

In [2]:
from faster_whisper import WhisperModel, BatchedInferencePipeline

model = WhisperModel("models\\tugstugi_ct2_float16", device="cuda", compute_type="float16")

batched_model = BatchedInferencePipeline(model=model)

print("Loaded batched model successfully")

Loaded batched model successfully


In [4]:
run_transcription(
    batched_model=batched_model,
    model_name="Faster-whisper-tugstugi",
    language='bn',
    vad_parameters={"min_silence_duration_ms": 4000, "speech_pad_ms": 5000}
)



Transcribing Rain with Faster-whisper-tugstugi...
Streaming Chunked Output:
0.00 - 2.20: দক্ষিণবঙ্গে আজ ও কাল ভারী
2.20 - 4.54: বৃষ্টির সম্ভাবনা কলকাতায় দফায় দফায়
4.54 - 6.86: বৃষ্টির সম্ভাবনা থাকছে। শুক্রবার থেকে
6.86 - 9.24: এই বৃষ্টি কমবে কলকাতায়। দুই
9.24 - 10.90: চব্বিশ পরগনায় ভারী বৃষ্টির হলুদ
10.90 - 12.66: সতর্কতা রয়েছে। ভারী বৃষ্টির সতর্কতা
12.66 - 15.40: থাকছে পূর্ব মেদিনীপুর, পশ্চিম বর্ধমান
15.40 - 19.14: বাঁকুড়াতেও। অন্য জেলাতেও বিক্ষিপ্ত দু
19.14 - 21.60: এক ফসলা বৃষ্টির সম্ভাবনা থাকছে।
21.60 - 24.66: শনি এবং রবিবার দক্ষিণবঙ্গে বৃষ্টির
24.66 - 29.18: পরিমাণ কমবে। আর অন্যদিকে উত্তরবঙ্গে
29.18 - 34.50: আগামীকাল থেকে বৃষ্টির পরিমাণ কমবে।
34.50 - 36.28: অয়ন ঘোষাল আমাদের প্রতিনিধিরে অইছেন
36.28 - 37.56: আমাদের সঙ্গে অয়ন সব মিলিয়ে
37.56 - 39.30: কি পূর্বাভাস হাওয়া অফিসে।
43.93 - 46.79: এই মুহূর্তে মৌসুমী অক্ষরেখা মুর্শিদাবাদ
46.79 - 49.47: থেকে গাঙ্গেয় দক্ষিণবঙ্গের প্রায় ওপোট
49.47 - 51.53: দিয়ে বাংলাদেশ পর্যন্ত বিস্তৃত সেই
51.53 - 53.49: মৌসুমী অক্ষরেখার হাত ধরে যে
53.49 - 55.

,Video,WER %,CER %,Word Accuracy %,GT Words,Pred Words,Correct Words,Substitutions,Deletions (Missed),Insertions
0,Rain,24.65,11.94,75.81,215,203,163,39,13,1
1,Delhi Protest,23.12,7.28,79.03,186,185,147,34,5,4
2,Subhendu CM,32.27,15.63,69.97,626,587,438,135,53,14
3,Dengue,31.59,11.87,70.88,364,367,258,100,6,9
4,Terrorist(Long),31.58,14.04,71.94,2527,2482,1818,575,134,89
5,Taslima Nasrin(Long),30.11,14.27,74.49,2670,2672,1989,560,121,123
6,Shamik(Long),33.94,13.27,70.10,2625,2660,1840,714,71,106
7,10am News(Long),32.74,13.43,69.99,2456,2383,1719,597,140,67


### Faster-whisper-bitwisemind

In [2]:
from faster_whisper import WhisperModel, BatchedInferencePipeline

model = WhisperModel("models\\bitwisemind_sam_ct2_float16", device="cuda", compute_type="float16")

batched_model = BatchedInferencePipeline(model=model)

print("Loaded batched model successfully")

Loaded batched model successfully


In [4]:
run_transcription(
    batched_model=batched_model,
    model_name="Faster-whisper-bitwisemind",
    language='bn',
    vad_parameters={"min_silence_duration_ms": 4000, "speech_pad_ms": 5000}
)



Transcribing Rain with Faster-whisper-bitwisemind...
Streaming Chunked Output:
0.00 - 2.20: দক্ষিণবঙ্গে আজ ও কাল ভারী
2.20 - 4.58: বৃষ্টির সম্ভাবনা কলকাতায় দফায় দফায়
4.58 - 6.86: বৃষ্টির সম্ভাবনা থাকছে শুক্রবার থেকে
6.86 - 9.14: এই বৃষ্টি কমবে কলকাতায় দুই
9.14 - 10.90: চব্বিশ পরগনায় ভারী বৃষ্টির হলুদ
10.90 - 12.66: সতর্কতা রয়েছে ভারী বৃষ্টির সতর্কতা
12.66 - 15.42: থাকছে পূর্ব মেদিনীপুর পশ্চিম বর্ধমান
15.42 - 19.22: বাঁকুড়াতেও অন্য জেলাতেও বিক্ষিপ্ত দুই
19.22 - 21.70: এক পশলা বৃষ্টির সম্ভাবনা থাকছে
21.70 - 24.68: শনি এবং রবিবার দক্ষিণবঙ্গে বৃষ্টির
24.68 - 29.14: পরিমাণ কমবে আর অন্যদিকে উত্তরবঙ্গে
29.14 - 34.78: আগামীকাল থেকে বৃষ্টির পরিমাণ কমবে
34.78 - 36.30: অয়ন ঘোষাল আমাদের প্রতিনিধি রয়েছেন
36.30 - 37.54: আমাদের সঙ্গে অয়ন সব মিলিয়ে
37.54 - 39.18: কি পূর্বাভাস হাওয়া অফিসে
43.89 - 46.81: এই মুহূর্তে মৌসুমী অক্ষরেখা মুর্শিদাবাদ
46.81 - 49.37: থেকে গাঙ্গেয় দক্ষিণবঙ্গের প্রায় ওপর
49.37 - 51.53: দিয়ে বাংলাদেশ পর্যন্ত বিস্তৃত সেই
51.53 - 53.49: মৌসুমে অক্ষরেখার হাত ধরে যে
53.49

,Video,WER %,CER %,Word Accuracy %,GT Words,Pred Words,Correct Words,Substitutions,Deletions (Missed),Insertions
0,Rain,16.74,10.06,83.72,215,203,180,22,13,1
1,Delhi Protest,16.67,4.91,86.02,186,184,160,19,7,5
2,Subhendu CM,17.25,11.71,84.98,626,580,532,34,60,14
3,Dengue,14.84,6.58,86.26,364,362,314,44,6,4
4,Terrorist(Long),19.63,9.58,83.18,2527,2477,2102,304,121,71
5,Taslima Nasrin(Long),18.99,11.97,83.48,2670,2517,2229,222,219,66
6,Shamik(Long),20.19,8.96,83.39,2625,2659,2189,376,60,94
7,10am News(Long),20.64,9.90,82.04,2456,2405,2015,324,117,66
